# Automatic broadcast football analysis
Experimental offline analysis. Enable a GPU in Colab (primary) or Kaggle. Free GPU availability is not guaranteed. Copy results to persistent storage; notebook disks disappear. Runtime is automatic; labels are used only for evaluation.

In [ ]:
import os, subprocess, sys
from pathlib import Path
import torch
assert sys.version_info >= (3,12), 'Use Python 3.12 or newer'
assert torch.cuda.is_available(), 'Allocate a GPU runtime first'
print(torch.__version__, torch.cuda.get_device_name(0))
ROOT=Path('/content/football') if Path('/content').exists() else Path('/kaggle/working/football')
REPO_REF='codex/match-analysis-20260930' # Replace with reviewed commit SHA after merging.
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/RockENZO/ML-object-detection-project',str(ROOT)],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout',REPO_REF],check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements-analysis.txt'],check=True)
subprocess.run([sys.executable,'tools/setup_pitch.py'],check=True)


## Input and durable output
In Colab, mount Google Drive and provide an MP4 plus the trained four-class checkpoint. In Kaggle use a permitted input dataset. The checkpoint is not a generic COCO model. Copy the supplied config/metadata templates and set known pitch dimensions/directions. Save shot artifacts directly to durable storage in Colab. Kaggle working outputs must be saved before the session ends.

In [ ]:
# Optional Colab mount:
# from google.colab import drive
# drive.mount('/content/drive')
VIDEO=Path('/content/drive/MyDrive/football/match.mp4')
CHECKPOINT=Path('/content/drive/MyDrive/football/best.pt')
BALL_CHECKPOINT=None # Optional Path to the reproduced broadcast-adapted four-class weights.
OUTPUT=Path('/content/drive/MyDrive/football/analysis')
# Kaggle alternatives:
# VIDEO=Path('/kaggle/input/my-match/match.mp4')
# CHECKPOINT=Path('/kaggle/input/my-model/best.pt')
# OUTPUT=Path('/kaggle/working/analysis')
CONFIG=ROOT/'configs/match.json'
METADATA=None # Set a metadata JSON file when verified directions are known.
assert VIDEO.is_file() and CHECKPOINT.is_file()


## Benchmark this allocated GPU first
Measure a 30-second clip using exactly the same settings as the full run. Save the resulting manifest and GPU model. Startup is included. Do not extrapolate full-match processing until representative shots, cuts and replays have been benchmarked.

In [ ]:
benchmark=OUTPUT.parent/'benchmark-30s'
cmd=[sys.executable,'-m','match_analysis','analyze','--video',str(VIDEO),'--checkpoint',str(CHECKPOINT),'--config',str(CONFIG),'--output',str(benchmark),'--device','0','--max-seconds','30']
if BALL_CHECKPOINT:cmd += ['--ball-checkpoint',str(BALL_CHECKPOINT)]
if METADATA:cmd += ['--metadata',str(METADATA)]
if benchmark.exists():cmd += ['--resume']
subprocess.run(cmd,check=True)
import json
manifest=json.loads((benchmark/'manifest.json').read_text())
print({k:manifest[k] for k in ('processed_samples','compute_seconds','processed_samples_per_compute_second')})


## Full run and recovery
Interrupted jobs resume from the last durable camera-shot boundary. The active shot is reprocessed, so output never bridges missing segments. Identical video/model/config/metadata/code hashes are required. Download the complete artifact directory to your local computer together with the same MP4. Run Streamlit locally; it needs no GPU.

In [ ]:
cmd=[sys.executable,'-m','match_analysis','analyze','--video',str(VIDEO),'--checkpoint',str(CHECKPOINT),'--config',str(CONFIG),'--output',str(OUTPUT),'--device','0']
if BALL_CHECKPOINT:cmd += ['--ball-checkpoint',str(BALL_CHECKPOINT)]
if METADATA:cmd += ['--metadata',str(METADATA)]
if OUTPUT.exists():cmd += ['--resume']
subprocess.run(cmd,check=True)
print(json.loads((OUTPUT/'summary.json').read_text()))


## Local viewing
```bash
python -m pip install -r requirements-dashboard.txt
python -m streamlit run dashboard/app.py -- --run /path/to/analysis
```
Select the local MP4 when the cloud path differs. The video controls drive the pitch and chart clock. Predictions are hollow/dashed. Team/ball coverage and unknown time must accompany any reported metrics. Follow `docs/MATCH_ANALYSIS.md` to freeze match partitions and evaluate independently annotated events; no acceptance claim is made by running this notebook.

## Broadcast adaptation and independent labels
Follow docs/MATCH_ANALYSIS.md to freeze development/validation/evaluation source matches and reproduce the adapted ball checkpoint. Pass --ball-checkpoint to the same CLI when using separate person and ball weights. Coaching geometry describes observed visible groups only. Annotate control changes and eligible/replay intervals independently using dashboard/review_annotations.py; no temporal accuracy claim is valid without those human labels. Local MPS results are not Colab throughput measurements.